In [67]:
from vertex_lite.custom import load_data,load_data_v2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import glob
import os
import numpy as np

In [68]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

def filter_data(time, pcts=[0.05, 0.25], noises=[0.2, 0.45, 0.7], local=False, cluster=False,summary=False):
    rpath = "resultados_cluster" if cluster else "resultados"

    dfs = []
    for noise in noises:
        for pct in pcts:
            dfs.append(
                load_data(
                    f"{rpath}/noise_{noise}/pct_{pct}/sim_*/cells.parquet",
                    columns=["time", "n_lados", "neighbor_of_mutant", "cell_id", "alive"],
                    time_filter=time
                )
            )

    df = pd.concat(dfs, ignore_index=True)

    if df.empty:
        print("No data found")
        return pd.DataFrame()

    df = df[df["alive"] == 1].copy()

    if local:
        df = df[df["neighbor_of_mutant"] == True]
    else:
        df = df.drop(columns=["neighbor_of_mutant"])

    df = (
        df[df["n_lados"] >= 3]
        .groupby(["time", "mesh_type", "pct_mutant", "sim_id", "n_lados"])
        .size()
        .reset_index(name="n_cells")
    )

    df["total_cells"] = (
        df.groupby(["time", "mesh_type", "pct_mutant", "sim_id"])["n_cells"]
        .transform("sum")
    )
    df["frac_cells"] = df["n_cells"] / df["total_cells"]

    stats = (
        df.groupby(["time", "mesh_type", "pct_mutant", "n_lados"])["frac_cells"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    stats["sem"] = stats["std"] / np.sqrt(stats["count"])

    if summary:
        return stats
    else:
        return df




In [69]:
filter_data(time=0,pcts=[0.05, 0.25], noises=[0.2, 0.45, 0.7])

,time,mesh_type,pct_mutant,sim_id,n_lados,n_cells,total_cells,frac_cells
0,0,0.2,0.05,0,4,1,400,0.0025
1,0,0.2,0.05,0,5,46,400,0.1150
2,0,0.2,0.05,0,6,308,400,0.7700
3,0,0.2,0.05,0,7,43,400,0.1075
4,0,0.2,0.05,0,8,1,400,0.0025
...,...,...,...,...,...,...,...,...
335,0,0.7,0.25,9,5,110,400,0.2750
336,0,0.7,0.25,9,6,157,400,0.3925
337,0,0.7,0.25,9,7,90,400,0.2250
338,0,0.7,0.25,9,8,24,400,0.0600


In [70]:
def plotting(pct, noise,local, cluster):
    tend = 4995

    raw = pd.concat(
        [
            filter_data(time=0,pcts=pct,noises=noise, local=local, cluster=cluster, summary=False).assign(label="t0"),
            filter_data(time=tend,pcts=pct,noises=noise,local=local, cluster=cluster, summary=False).assign(label="tend"),
        ],
        ignore_index=True
    )

    stats = pd.concat(
        [
            filter_data(time=0,pcts=pct,noises=noise, local=local, cluster=cluster, summary=True).assign(label="t0"),
            filter_data(time=tend,pcts=pct,noises=noise, local=local, cluster=cluster, summary=True).assign(label="tend"),
        ],
        ignore_index=True
    )

    sns.set_theme(style="white")
    g = sns.FacetGrid(
        raw,
        row="mesh_type",
        col="pct_mutant",
        height=3,
        aspect=1.4,
        sharex=True,
        sharey=True,
        margin_titles=True
    )

    def draw_panel(data, **kwargs):
        ax = plt.gca()

        mesh = data["mesh_type"].iloc[0]
        pct = data["pct_mutant"].iloc[0]

        raw_sub = raw[(raw["mesh_type"] == mesh) & (raw["pct_mutant"] == pct)]
        stats_sub = stats[(stats["mesh_type"] == mesh) & (stats["pct_mutant"] == pct)]

        # puntos individuales
        for label, offset in [("t0", -0.12), ("tend", 0.12)]:
            d = raw_sub[raw_sub["label"] == label]
            x = d["n_lados"].astype(float).to_numpy() + offset
            y = d["frac_cells"].to_numpy()

            ax.scatter(
                x, y,
                s=18,
                alpha=0.35,
                color="gray"
            )

        # media ± sem
        colors = {"t0": "#2166ac", "tend": "#b2182b"}
        for label, offset in [("t0", -0.12), ("tend", 0.12)]:
            d = stats_sub[stats_sub["label"] == label]
            x = d["n_lados"].astype(float).to_numpy() + offset
            y = d["mean"].to_numpy()
            yerr = d["sem"].to_numpy()

            ax.errorbar(
                x, y, yerr=yerr,
                fmt="_",
                linestyle="none",
                color=colors[label],
                capsize=3,
                markersize=18,
                elinewidth=1.2
            )

        ax.grid(False)

    g.map_dataframe(draw_panel)
    g.set_titles(
    row_template="D = {row_name}",
    col_template="% Mutants = {col_name}"
)
    for ax in g.axes.flat:
        nrows, ncols = g.axes.shape

        for i, row_axes in enumerate(g.axes):
            for j, ax in enumerate(row_axes):
                ax.set_ylim([-0.05, 0.85])
                ax.set_xlim(2.5, 11.5)
                ax.set_xticks([3,4,5,6,7,8,9,10,11])

                scope = "Local" if local else "Global"

                ax.spines["top"].set_visible(False)
                ax.spines["right"].set_visible(False)

                ax.tick_params(
                    axis="both", which="both", direction="out",
                    length=4, labelsize=13, width=1
                )

                ax.tick_params(axis="x", rotation=45)

                ax.xaxis.set_ticks_position("bottom")
                ax.yaxis.set_ticks_position("left")

                # solo labels abajo
                if i == nrows - 1:
                    ax.set_xlabel("Number of Neighbours")
                    ax.tick_params(axis="x", labelbottom=True)
                else:
                    ax.set_xlabel("")
                    ax.tick_params(axis="x", labelbottom=False)

                # solo labels izquierda
                if j == 0:
                    ax.set_ylabel("Fraction of cells")
                    ax.tick_params(axis="y", labelleft=True)
                else:
                    ax.set_ylabel("")
                    ax.tick_params(axis="y", labelleft=False)
                

    mutant_dist = "clustered" if cluster else "uniform"
    
    # g.figure.suptitle(
    #     f"{scope} distribution of polygon side for {mutant_dist} mutants",
    #     y=1.02
    # )

    savedir = Path("figures_paper_final")  / "histogram_zoom"
    savedir.mkdir(parents=True, exist_ok=True)

    filename = savedir / f"Hist_{'Cluster' if cluster else 'uniform'}_{scope}_pct{pct}_noise{noise}.svg"
    plt.savefig(filename, format="svg", bbox_inches="tight")
    plt.close()

In [71]:
def plotting_violin(local,cluster):
    tend= 4995 if cluster else 4995
    combined = pd.concat(
        [filter_data(time=0,local=local,cluster=cluster).assign(time='t0'), filter_data(time=tend,local=local,cluster=cluster).assign(time='tend')],
        ignore_index=True
    )

    g = sns.catplot(
        data=combined,
        x="n_lados",
        y="frac_cells",
        hue="time",
        col="pct_mutant",
        row="mesh_type",
        kind="violin",
        split=True,
        inner="quartile",
        cut=0,
        height=3,
        aspect=1.5,
    )
    
    from pathlib import Path
   #Donde guardar y guardar

    scope = "Local" if local else "Global"
    mutant_dist = "clustered" if cluster else "uniform"

    g.figure.suptitle(
        f"{scope} distribution of polygon side for {mutant_dist} mutants",
        y=1.02
    )

    savedir = Path("figures_paper_final")  / "histogram_zoom"
    savedir.mkdir(parents=True, exist_ok=True)

    filename = savedir / f"Hist_{'Cluster' if cluster else 'uniform'}_{scope}.svg"
    plt.savefig(filename, format="svg", bbox_inches="tight")
    plt.show()

In [72]:
pcts=[0.05, 0.25]
noises=[0.2, 0.7]
for local in [True, False]:
    for cluster in [True, False]:

        plotting(pct=pcts,noise=noises, local=local, cluster=cluster)